In [4]:
import numpy as np

raw_pat１ = "/root/AIMET_ENV/tools/ruri/convert/netrun/seq_genie.raw"   # 読み込む raw ファイルのパス
#raw_path = "/root/AIMET_ENV/tools/ruri/convert/netrun/pool_genie.raw"   # 読み込む raw ファイルのパス

raw_path = "/root/AIMET_ENV/tools/ruri/convert/netrun/out/Result_0/sequence_output.raw"   # 読み込む raw ファイルのパス
dtype = np.float32      # 必要に応じて np.uint8 / np.int16 などに変更

vector = np.fromfile(raw_path, dtype=dtype)
print("vector shape:", vector.shape)

vector shape: (393216,)


In [7]:
raw_path1 = "/root/AIMET_ENV/tools/ruri/convert/netrun/seq_genie.raw"
raw_path2 = "/root/AIMET_ENV/tools/ruri/convert/netrun/out/Result_0/sequence_output.raw"

vector1 = np.fromfile(raw_path1, dtype=np.float32)
vector2 = np.fromfile(raw_path2, dtype=np.float32)

n1 = vector1.shape[0] // 768
n2 = vector2.shape[0] // 768

matrix1 = vector1[:n1 * 768].reshape(n1, 768)
matrix2 = vector2[:n2 * 768].reshape(n2, 768)

print("matrix1 shape:", matrix1.shape)
print("matrix2 shape:", matrix2.shape)

print("matrix1[:14]", matrix1[:14])
print("matrix2[:14]", matrix2[:14])

matrix1 shape: (14, 768)
matrix2 shape: (512, 768)
matrix1[:14] [[-0.7627821  -0.37160265 -0.87739515 ... -1.3038127   0.48550385
  -1.3194742 ]
 [-0.63357544  0.75210387 -0.7990881  ... -0.4524013   0.43745178
  -0.3036179 ]
 [-0.6652542   0.79161334 -1.0343653  ... -0.12351161  0.7638499
  -0.38192496]
 ...
 [-1.024043    0.68910223 -0.4228582  ... -0.45418102  0.3342288
  -0.10251108]
 [-0.53925097  0.88700557 -1.217675   ... -0.7442731   0.25058264
  -0.38512844]
 [-0.2762104   0.62360907 -0.7172216  ... -1.0916718   0.54565793
  -0.6791359 ]]
matrix2[:14] [[-6.3001603e-02  6.3001603e-02 -9.6104133e-01 ... -1.4643422e+00
   7.9873216e-01 -8.2151240e-01]
 [-4.6272362e-03 -7.9730839e-02 -9.0836203e-01 ... -1.5330390e+00
   8.4927583e-01 -8.2720745e-01]
 [-7.1188249e-04 -4.6628304e-02 -8.7703919e-01 ... -1.5490563e+00
   8.3076686e-01 -8.3752972e-01]
 ...
 [-1.9576769e-02 -2.8831240e-02 -8.6707288e-01 ... -1.6373297e+00
   7.9410493e-01 -8.4073323e-01]
 [-1.4593591e-02 -3.2746594e-02 

In [6]:
import os
import numpy as np
import onnx
import onnxruntime as ort

from transformers import AutoTokenizer

import aimet_onnx
from aimet_onnx.common.defs import QuantScheme
from aimet_onnx import QuantizationSimModel

#FP32_ONNX = "/root/AIMET_ruri/ruri-onnx/model_simplified.onnx"
#FP32_ONNX = "/root/AIMET_ENV/tools/ruri/quantize/ruri-export-onnx/model_fix.onnx"
FP32_ONNX = "/root/AIMET_ruri/ruri-onnx/model.onnx"  # ← 適宜変更
OUT_DIR = "/root/AIMET_ENV/tools/ruri/model"

os.makedirs(OUT_DIR, exist_ok=True)

MODEL_ID = "/root/AIMET_ruri/ruri-small-v2"
# tokenizer は事前にロード
tokenizer = AutoTokenizer.from_pretrained(
    "/root/AIMET_ruri/ruri-small-v2",
    trust_remote_code=True,
)

# 1) load onnx
model = onnx.load_model(FP32_ONNX)

# 2) (推奨) simplify

import onnx


model = onnx.load_model(FP32_ONNX)

def fix_input_shapes(model, shapes_dict):
    for inp in model.graph.input:
        name = inp.name
        if name in shapes_dict:
            shape = shapes_dict[name]
            for i, dim in enumerate(inp.type.tensor_type.shape.dim):
                dim.dim_value = shape[i]

    return model

""""
model = fix_input_shapes(
    model,
    {
        "input_ids":      [1, 512],
        "attention_mask": [1, 512],
        #"token_type_ids": [1, 512],
        #"position_ids":   [1, 512],
    }
)
 """
# 3) create QuantSim (CPU only, W8/A16)
providers = ["CPUExecutionProvider"]
sim = QuantizationSimModel(

    model,
    param_type=aimet_onnx.int8,
    activation_type=aimet_onnx.int16,
    #quant_scheme=QuantScheme.min_max,
    quant_scheme=QuantScheme.post_training_tf_enhanced,
    #config_file="default",
    config_file="htp_v73",
    #config_file="/root/AIMET_ruri_v2/tool/quantsim_fp_escape.json",  # ← 適宜変更
    providers=providers,
)



2026-05-14 04:58:32,427 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:V73


In [7]:
# 入力テキスト（適宜変更）
text = "クエリ: 瑠璃色はどんな色？"

# tokenize（model の固定入力 shape [1, 512] に合わせる）
enc = tokenizer(
    text,
    return_tensors="np",
    #padding="max_length",
    truncation=True,
    max_length=512,
)

# ONNX model から推論セッションを作成して実行
session = ort.InferenceSession(model.SerializeToString(), providers=["CPUExecutionProvider"])

feed = {}
for inp in session.get_inputs():
    name = inp.name
    if name in enc:
        feed[name] = enc[name].astype(np.int64)
    elif name == "token_type_ids":
        feed[name] = np.zeros_like(enc["input_ids"], dtype=np.int64)
    elif name == "position_ids":
        seq_len = enc["input_ids"].shape[1]
        feed[name] = np.arange(seq_len, dtype=np.int64)[None, :]
    else:
        raise KeyError(f"Unsupported input name: {name}")

outputs = session.run(None, feed)

# 出力確認
for i, out in enumerate(outputs):
    print(f"output[{i}] shape: {out.shape}, dtype: {out.dtype}")
    print(f"output[{i}] first 10 values: {out.reshape(-1)[:10]}")

Fail: [ONNXRuntimeError] : 1 : FAIL : Fatal error: aimet.customop.cpu:QcQuantizeOp(-1) is not a registered function/op

In [2]:
import numpy as np
raw_path1 = "/root/AIMET_ENV/tools/ruri/convert/sim_output.raw"   # 読み込む raw ファイルのパス

raw_path2 = "/root/AIMET_ENV/tools/ruri/convert/seq_output.raw"   # 読み込む raw ファイルのパス


# rawファイルを読み込んでコサイン類似度を計算
vec1 = np.fromfile(raw_path1, dtype=np.float32)
vec2 = np.fromfile(raw_path2, dtype=np.float32)

# 1) フラット同士（長さを揃えて比較）
min_len = min(vec1.size, vec2.size)
a = vec1[:min_len]
b = vec2[:min_len]

eps = 1e-12
cos_flat = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps)
print("cosine similarity (flat):", float(cos_flat))

# 2) 768次元ごとの行単位で比較（先頭から揃う分だけ）
row_count = min(vec1.size, vec2.size) // 768
m1 = vec1[:row_count * 768].reshape(row_count, 768)
m2 = vec2[:row_count * 768].reshape(row_count, 768)

dot = np.sum(m1 * m2, axis=1)
norm = np.linalg.norm(m1, axis=1) * np.linalg.norm(m2, axis=1) + eps
cos_per_row = dot / norm

print("row_count:", row_count)
print("cosine similarity (per-row) first 10:", cos_per_row[:10])
print("cosine similarity (per-row) mean:", float(np.mean(cos_per_row)))

cosine similarity (flat): 0.889293372631073
row_count: 1
cosine similarity (per-row) first 10: [0.8892934]
cosine similarity (per-row) mean: 0.889293372631073


In [45]:
FP32_ONNX = "/root/AIMET_ENV/tools/ruri/quantize/ruri-export-onnx/model_fix_mask.onnx" 
onnx_model = onnx.load(FP32_ONNX)
session = ort.InferenceSession(
    onnx_model.SerializeToString(),
    providers=["CPUExecutionProvider"],
)

text = "クエリ: 瑠璃色はどんな色？"
enc = tokenizer(
    text,
    return_tensors="np",
    truncation=True,
    max_length=512,
    paading=False
)

feed = {}
for inp in session.get_inputs():
    name = inp.name
    if name in enc:
        feed[name] = enc[name].astype(np.int64)
    elif name == "token_type_ids":
        feed[name] = np.zeros_like(enc["input_ids"], dtype=np.int64)
    elif name == "position_ids":
        seq_len = enc["input_ids"].shape[1]
        feed[name] = np.arange(seq_len, dtype=np.int64)[None, :]
    else:
        raise KeyError(f"Unsupported input name: {name}")

outputs = session.run(None, feed)

for i, out in enumerate(outputs):
    print(f"output[{i}] shape: {out.shape}, dtype: {out.dtype}")
    print(out.reshape(-1)[:20])





output[0] shape: (1, 14, 768), dtype: float32
[-1.3951111   0.19274704 -0.47006088 -0.54013145  1.0519888  -0.71940494
  0.2962852   0.43273148 -1.103201    0.8304254   0.41491103 -0.3792354
 -0.5797487   0.46798992  1.1597687  -0.17563583 -0.10857006 -0.18023285
  0.00250669  0.5707231 ]
output[1] shape: (1, 768), dtype: float32
[-1.2267492   0.2258042  -0.6635575  -0.50708556  1.002064   -0.024027
 -0.11228243  0.37909132 -0.877453    0.46234664  0.53016216 -0.5570603
 -0.3250673   0.5650288   1.2038639  -0.5702661  -0.27419144  0.09386112
  0.08312891  0.9352729 ]


In [46]:
FP32_ONNX = "/root/AIMET_ENV/tools/ruri/quantize/ruri-export-onnx/model_fix_mask.onnx" 
onnx_model = onnx.load(FP32_ONNX)


def fix_input_shapes(model, shapes_dict):
    for inp in model.graph.input:
        name = inp.name
        if name in shapes_dict:
            shape = shapes_dict[name]
            for i, dim in enumerate(inp.type.tensor_type.shape.dim):
                dim.dim_value = shape[i]

    return model


onnx_model = fix_input_shapes(
    onnx_model,
    {
        "input_ids":      [1, 512],
        "attention_mask": [1, 512],
        "token_type_ids": [1, 512],
        "position_ids":   [1, 512],
    }
)
 

session = ort.InferenceSession(
    onnx_model.SerializeToString(),
    providers=["CPUExecutionProvider"],
)

text = "クエリ: 瑠璃色はどんな色？"
enc = tokenizer(
    text,
    padding="max_length",
    return_tensors="np",
    truncation=True,
    max_length=512,
)

feed = {}
for inp in session.get_inputs():
    name = inp.name
    if name in enc:
        feed[name] = enc[name].astype(np.int64)
    elif name == "token_type_ids":
        feed[name] = np.zeros_like(enc["input_ids"], dtype=np.int64)
    elif name == "position_ids":
        seq_len = enc["input_ids"].shape[1]
        feed[name] = np.arange(seq_len, dtype=np.int64)[None, :]
    else:
        raise KeyError(f"Unsupported input name: {name}")

outputs2 = session.run(None, feed)

for i, out in enumerate(outputs2):
    print(f"output[{i}] shape: {out.shape}, dtype: {out.dtype}")
    print(out.reshape(-1)[:10])

#print(outputs2[0])

output[0] shape: (1, 512, 768), dtype: float32
[-1.3951107   0.1927475  -0.4700608  -0.54013216  1.0519882  -0.71940553
  0.29628488  0.43273023 -1.1032009   0.8304253 ]
output[1] shape: (1, 768), dtype: float32
[-1.2267495   0.2258047  -0.6635576  -0.50708586  1.002064   -0.02402702
 -0.11228251  0.37909108 -0.87745273  0.46234706]


In [47]:
import numpy as np


def cosine(a, b):
    a = a.reshape(-1)
    b = b.reshape(-1)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

cos = cosine(outputs[1], outputs2[1])
print("cosine similarity:", cos)


cosine similarity: 1.0


In [ ]:
graph = onnx_model.graph

def _shape_from_value_info(value_info):
    t = value_info.type
    if not t.HasField("tensor_type"):
        return "non-tensor"
    dims = []
    for d in t.tensor_type.shape.dim:
        if d.HasField("dim_value"):
            dims.append(d.dim_value)
        elif d.HasField("dim_param"):
            dims.append(d.dim_param)  # 例: "batch", "sequence"
        else:
            dims.append("?")
    return dims

print(f"ONNX: {FP32_ONNX}")

print("\n[Inputs]")
for vi in graph.input:
    print(f"- {vi.name}: {_shape_from_value_info(vi)}")

print("\n[Outputs]")
for vi in graph.output:
    print(f"- {vi.name}: {_shape_from_value_info(vi)}")